# Herleitung der Dynamik des vereinfachten Scara Roboters mit dem Lagrange-Formalismus

Jonas Frei, 23.09.2026, jonas.frei@ost.ch

In [ ]:
import sympy as sp
from IPython.display import display

sp.init_printing()

## Variablendeklaration

In [ ]:
l1, lc1, lc2, m1, m2 = sp.symbols('l1 lc1 lc2 m1 m2')
Ixx1, Ixy1, Ixz1, Iyy1, Iyz1, Izz1 = sp.symbols('Ixx1 Ixy1 Ixz1 Iyy1 Iyz1 Izz1')
Ixx2, Ixy2, Ixz2, Iyy2, Iyz2, Izz2 = sp.symbols('Ixx2 Ixy2 Ixz2 Iyy2 Iyz2 Izz2')

t = sp.symbols('t')
q1 = sp.Function('q1')(t)
q2 = sp.Function('q2')(t)

## Körper 1

Berechnung der Vorwärtskinematik bis zum Massenmittelpunkt

In [ ]:
r01x = lc1*sp.cos(q1)
r01y = lc1*sp.sin(q1)
r01z = sp.Integer(0)
R01 = sp.Matrix([[sp.cos(q1), -sp.sin(q1), 0],
                 [sp.sin(q1),  sp.cos(q1), 0],
                 [0, 0, 1]])

Berechnung der Winkelgeschwindigkeiten

In [ ]:
dR01 = sp.diff(R01, t)

om01 = sp.simplify(dR01*R01.T)

omega01 = sp.Matrix([om01[2, 1], om01[0, 2], om01[1, 0]])

omega1 = R01.T*omega01

Berechnung des Geschwindigkeitsvektor

In [ ]:
v1 = sp.Matrix.vstack(
    sp.Matrix([sp.diff(r01x, t), sp.diff(r01y, t), sp.diff(r01z, t)]),
    omega1
)

Berechnung der kinetischen und potentiellen Energie

In [ ]:
M1 = sp.Matrix([[m1, 0, 0, 0, 0, 0],
                [0, m1, 0, 0, 0, 0],
                [0, 0, m1, 0, 0, 0],
                [0, 0, 0, Ixx1, Ixy1, Ixz1],
                [0, 0, 0, Ixy1, Iyy1, Iyz1],
                [0, 0, 0, Ixz1, Iyz1, Izz1]])

K1 = sp.simplify((sp.Rational(1, 2)*v1.T*M1*v1)[0, 0])
P1 = (m1*sp.Matrix([[0, 0, -sp.Float(9.81)]])*sp.Matrix([r01x, r01y, r01z]))[0, 0]

display(sp.Eq(sp.Symbol('K_1'), K1, evaluate=False))
display(sp.Eq(sp.Symbol('P_1'), P1, evaluate=False))

## Körper 2

Berechnung der Vorwärtskinematik bis zum Massenmittelpunkt

In [ ]:
r02x = l1*sp.cos(q1) + lc2*sp.cos(q1 + q2)
r02y = l1*sp.sin(q1) + lc2*sp.sin(q1 + q2)
r02z = sp.Integer(0)
R02 = sp.Matrix([[sp.cos(q1 + q2), -sp.sin(q1 + q2), 0],
                 [sp.sin(q1 + q2),  sp.cos(q1 + q2), 0],
                 [0, 0, 1]])
dR02 = sp.diff(R02, t)

Berechnung der Winkelgeschwindigkeiten

In [ ]:
om02 = sp.simplify(dR02*R02.T)

omega02 = sp.Matrix([om02[2, 1], om02[0, 2], om02[1, 0]])

omega2 = R02.T*omega02

Berechnung des Geschwindigkeitsvektor

In [ ]:
v2 = sp.Matrix.vstack(
    sp.Matrix([sp.diff(r02x, t), sp.diff(r02y, t), sp.diff(r02z, t)]),
    omega2
)

Berechnung der kinetischen und potentiellen Energie

In [ ]:
M2 = sp.Matrix([[m2, 0, 0, 0, 0, 0],
                [0, m2, 0, 0, 0, 0],
                [0, 0, m2, 0, 0, 0],
                [0, 0, 0, Ixx2, Ixy2, Ixz2],
                [0, 0, 0, Ixy2, Iyy2, Iyz2],
                [0, 0, 0, Ixz2, Iyz2, Izz2]])

K2 = sp.simplify((sp.Rational(1, 2)*v2.T*M2*v2)[0, 0])
P2 = (m2*sp.Matrix([[0, 0, -sp.Float(9.81)]])*sp.Matrix([r02x, r02y, r02z]))[0, 0]

display(sp.Eq(sp.Symbol('K_2'), K2, evaluate=False))
display(sp.Eq(sp.Symbol('P_2'), P2, evaluate=False))

## Berechnung der Summen der kinetischen und potentiellen Energien aller Körper

In [ ]:
K = sp.simplify(K1 + K2)
display(sp.Eq(sp.Symbol('K'), K, evaluate=False))

P = sp.simplify(P1 + P2)
display(sp.Eq(sp.Symbol('P'), P, evaluate=False))

## Berechnung der Gelenkmomente

In [ ]:
dq1 = sp.diff(q1, t)
dq2 = sp.diff(q2, t)

Q1 = sp.simplify(sp.diff(sp.diff(K, dq1), t) - sp.diff(K, q1) + sp.diff(P, q1))
Q2 = sp.simplify(sp.diff(sp.diff(K, dq2), t) - sp.diff(K, q2) + sp.diff(P, q2))

q1dd = sp.diff(q1, (t, 2))
q2dd = sp.diff(q2, (t, 2))

Q1 = sp.collect(Q1, [q1dd, q2dd, dq1, dq2])
Q2 = sp.collect(Q2, [q1dd, q2dd, dq1, dq2])

display(sp.Eq(sp.Symbol('Q_1'), Q1, evaluate=False))
display(sp.Eq(sp.Symbol('Q_2'), Q2, evaluate=False))

## Berechnung der Massenmatrix aus den Gelenkmomenten

In [ ]:
M11 = sp.simplify(sp.diff(Q1, q1dd))
M12 = sp.simplify(sp.diff(Q1, q2dd))
M21 = sp.simplify(sp.diff(Q2, q1dd))
M22 = sp.simplify(sp.diff(Q2, q2dd))

M = sp.Matrix([[M11, M12],
               [M21, M22]])

display(sp.Eq(sp.Symbol('M'), M, evaluate=False))

## Zahlen einsetzen

Annahme für die Trägheitsberechnung: Glieder lassen sich durch einen dünnen Stab modellieren

In [ ]:
l1_val, l2_val = 1, 1
lc1_val, lc2_val = l1_val/2, l2_val/2
m1_val, m2_val = 0.2, 0.1
Izz1_val = sp.Rational(1, 12)*m1_val*l1_val**2
Izz2_val = sp.Rational(1, 12)*m2_val*l2_val**2

subs_dict = {l1: l1_val, lc1: lc1_val, lc2: lc2_val,
             m1: m1_val, m2: m2_val, Izz1: Izz1_val, Izz2: Izz2_val}

M_num = sp.simplify(M.subs(subs_dict))
display(sp.Eq(sp.Symbol('M'), sp.N(M_num, 4), evaluate=False))